# Dataset Completeness Check

This notebook checks the completeness of the processed dataset manifest for the columns required for training/evaluation:
- **sample_id**, **dataset**, **split**, **label**, **pose_npz**, **optflow_npz**

In [1]:
import pandas as pd
from pathlib import Path

# Resolve paths: works when cwd is project root or notebooks/
ROOT = Path(".") if Path("data/processed/manifest.csv").exists() else Path("..")
MANIFEST_PATH = ROOT / "data" / "processed" / "manifest.csv"
COLS = ["sample_id", "dataset", "split", "label", "pose_npz", "optflow_npz"]

In [2]:
df = pd.read_csv(MANIFEST_PATH)
df_sub = df[COLS].copy()
print(f"Total rows: {len(df):,}")
df_sub.head()

Total rows: 5,649


,sample_id,dataset,split,label,pose_npz,optflow_npz
0,ipn_train_1CM1_4_R_#229_D0X_1_17,ipn_hand,train,D0X,data/processed/ipn_hand/train/ipn_train_1CM1_4...,data/processed/ipn_hand/train/ipn_train_1CM1_4...
1,ipn_val_1CM42_13_R_#143_D0X_1_9,ipn_hand,val,D0X,data/processed/ipn_hand/val/ipn_val_1CM42_13_R...,data/processed/ipn_hand/val/ipn_val_1CM42_13_R...
2,ipn_test_1CM1_1_R_#217_D0X_1_28,ipn_hand,test,D0X,data/processed/ipn_hand/test/ipn_test_1CM1_1_R...,data/processed/ipn_hand/test/ipn_test_1CM1_1_R...
3,ipn_train_1CM1_4_R_#229_G11_18_55,ipn_hand,train,G11,data/processed/ipn_hand/train/ipn_train_1CM1_4...,data/processed/ipn_hand/train/ipn_train_1CM1_4...
4,ipn_train_1CM1_4_R_#229_B0B_56_284,ipn_hand,train,B0B,data/processed/ipn_hand/train/ipn_train_1CM1_4...,data/processed/ipn_hand/train/ipn_train_1CM1_4...


## Train / Val / Test split sizes

In [12]:
split_counts = df_sub["split"].value_counts().sort_index()
total = len(df_sub)
print("Split sizes:")
for name in ["train", "val", "test"]:
    if name in split_counts.index:
        n = split_counts[name]
        pct = n / total * 100
        print(f"  {name:6s}: {n:,} ({pct:.1f}%)")
print(f"  {'Total':6s}: {total:,} (100.0%)")
# Show any other split values if present
other = split_counts.drop(["train", "val", "test"], errors="ignore")
if len(other):
    print("  Other:", dict(other))
split_counts

Split sizes:
  train : 3,646 (64.5%)
  val   : 393 (7.0%)
  test  : 1,610 (28.5%)
  Total : 5,649 (100.0%)


split
test     1610
train    3646
val       393
Name: count, dtype: int64

## 1. Missing values (NaN / empty string)

In [5]:
missing = df_sub.isna().sum()
empty = df_sub.apply(lambda s: (s.astype(str).str.strip() == "").sum())
stripped_empty = df_sub.apply(lambda s: s.astype(str).str.strip() == "")
incomplete_mask = df_sub.isna() | stripped_empty
total_incomplete = incomplete_mask.sum()
summary = pd.DataFrame({
    "missing_na": missing,
    "empty_str": empty,
    "total_incomplete": total_incomplete,
})
summary["complete"] = len(df_sub) - summary["total_incomplete"]
summary["pct_complete"] = (summary["complete"] / len(df_sub) * 100).round(2)
summary

,missing_na,empty_str,total_incomplete,complete,pct_complete
sample_id,0,0,0,5649,100.0
dataset,0,0,0,5649,100.0
split,0,0,0,5649,100.0
label,0,0,0,5649,100.0
pose_npz,0,0,0,5649,100.0
optflow_npz,0,0,0,5649,100.0


## 2. Rows with all 6 columns complete

In [6]:
def is_complete(series):
    return series.notna() & (series.astype(str).str.strip() != "")

all_complete = df_sub.apply(is_complete).all(axis=1)
n_complete = all_complete.sum()
n_incomplete = (~all_complete).sum()
print(f"Rows with all 6 columns complete: {n_complete:,} ({n_complete / len(df_sub) * 100:.2f}%)")
print(f"Rows with at least one missing/empty: {n_incomplete:,}")

Rows with all 6 columns complete: 5,649 (100.00%)
Rows with at least one missing/empty: 0


In [7]:
if n_incomplete > 0:
    incomplete = df_sub[~all_complete]
    print("\nSample of incomplete rows (first 10):")
    display(incomplete.head(10))

## 3. File existence (pose_npz, optflow_npz)

Paths in the manifest are relative to the project root. We check whether the files exist.

In [8]:
root = ROOT
pose_exists = df_sub["pose_npz"].dropna().apply(lambda p: (root / str(p).strip()).exists())
optflow_exists = df_sub["optflow_npz"].dropna().apply(lambda p: (root / str(p).strip()).exists())

pose_ok = pose_exists.sum()
pose_total = len(pose_exists)
optflow_ok = optflow_exists.sum()
optflow_total = len(optflow_exists)

pct_pose = (pose_ok / pose_total * 100) if pose_total else 0
pct_optflow = (optflow_ok / optflow_total * 100) if optflow_total else 0
print(f"pose_npz:   {pose_ok:,} / {pose_total:,} files exist ({pct_pose:.2f}%)")
print(f"optflow_npz: {optflow_ok:,} / {optflow_total:,} files exist ({pct_optflow:.2f}%)")

pose_npz:   5,649 / 5,649 files exist (100.00%)
optflow_npz: 5,649 / 5,649 files exist (100.00%)


In [9]:
# Among rows with non-null paths, how many have both files on disk?
root = ROOT
mask = df_sub["pose_npz"].notna() & (df_sub["pose_npz"].astype(str).str.strip() != "")
mask = mask & df_sub["optflow_npz"].notna() & (df_sub["optflow_npz"].astype(str).str.strip() != "")
pose_paths = df_sub.loc[mask, "pose_npz"].astype(str).str.strip()
optflow_paths = df_sub.loc[mask, "optflow_npz"].astype(str).str.strip()
pose_files_ok = pd.Series([(root / p).exists() for p in pose_paths], index=pose_paths.index)
optflow_files_ok = pd.Series([(root / p).exists() for p in optflow_paths], index=optflow_paths.index)
both_files_ok = pose_files_ok & optflow_files_ok
print(f"Rows where both pose_npz and optflow_npz files exist: {both_files_ok.sum():,} / {len(both_files_ok):,}")

Rows where both pose_npz and optflow_npz files exist: 5,649 / 5,649
